In [9]:
import sqlite3
import pandas as pd 

In [10]:
conn=sqlite3.connect("RetailIQ 360.db")
cursor=conn.cursor()
    

In [11]:
Retail_main = pd.read_sql('select * from RetailIQ360',conn)

In [3]:
fact_order_items=pd.read_sql('select * from fact_order_items',conn)

In [7]:
top_10_Cus_by_sales= pd.read_sql('''SELECT
    customer_id,
    SUM(sales) AS total_sales
FROM fact_order_items
GROUP BY customer_id
ORDER BY total_sales DESC
LIMIT 10;''',conn)

In [9]:
profitability_analysis = pd.read_sql('''SELECT
    product_id,
    SUM(profit) AS total_profit,
    SUM(sales) AS total_sales,
    ROUND(
        (SUM(profit) * 100.0 / SUM(sales)),
        2
    ) AS profit_margin_pct
FROM fact_order_items
GROUP BY product_id
ORDER BY total_profit DESC;''',conn)

In [11]:
rank_products_desc = pd.read_sql('''SELECT
    product_id,
    SUM(sales) AS total_sales,

    RANK() OVER (
        ORDER BY SUM(sales) DESC
    ) AS sales_rank

FROM fact_order_items
GROUP BY product_id;''',conn)

In [13]:
cus_order_frequency = pd.read_sql('''SELECT
    customer_id,
    COUNT(DISTINCT order_id) AS total_orders
FROM fact_order_items
GROUP BY customer_id
ORDER BY total_orders DESC;''',conn)

In [15]:
com_daily_sales = pd.read_sql('''SELECT
    order_date,
    SUM(sales) AS daily_sales,

    SUM(SUM(sales)) OVER (
        ORDER BY order_date
    ) AS cumulative_sales

FROM fact_order_items
GROUP BY order_date;''',conn)


In [16]:
moving_avg_7d = pd.read_sql('''SELECT
    order_date,
    SUM(sales) AS daily_sales,

    AVG(SUM(sales)) OVER (
        ORDER BY order_date
        ROWS BETWEEN 6 PRECEDING
        AND CURRENT ROW
    ) AS moving_avg_7d

FROM fact_order_items
GROUP BY order_date;''',conn)

# PARETO ANALYSIS START FROM HERE

In [17]:
pareto_analysis_table = pd.read_sql('''WITH product_sales AS (

    SELECT
        product_id,
        SUM(sales) AS total_sales
    FROM fact_order_items
    GROUP BY product_id

),

ranked AS (

    SELECT
        *,
        SUM(total_sales) OVER (
            ORDER BY total_sales DESC
        ) AS cumulative_sales,

        SUM(total_sales) OVER () AS grand_total

    FROM product_sales
)

SELECT
    *,
    ROUND(
        cumulative_sales * 100.0 / grand_total,
        2
    ) AS cumulative_pct
FROM ranked;''',conn)

# NOW FROM HERE WE ARE GONNA DO THE SIMPLE 'COHORT ANALYSIS'.

In [19]:
cohort_table = pd.read_sql('''SELECT
    customer_id,
    MIN(order_date) AS first_purchase_date
FROM fact_order_items
GROUP BY customer_id;''',conn)

# HIGH VALUE CUSTOMERS.

In [20]:
high_value_customers = pd.read_sql('''SELECT
    customer_id,
    SUM(sales) AS lifetime_value,

    CASE

        WHEN SUM(sales) >= 5000
            THEN 'High Value'

        WHEN SUM(sales) >= 2000
            THEN 'Medium Value'

        ELSE 'Low Value'

    END AS customer_segment

FROM fact_order_items
GROUP BY customer_id;''',conn)

# GEOGRAPHICAL ANALYTICS start from here

In [21]:
dim_location = pd.read_sql('''select * from dim_location''',conn)

In [24]:
dim_location

,Order City,Order State,Order Country,Order Region,Latitude,Longitude,location_id
0,Bekasi,Java Occidental,Indonesia,Southeast Asia,18.251453,-66.037056,1
1,Bikaner,Rajastán,India,South Asia,18.279451,-66.037064,2
2,Bikaner,Rajastán,India,South Asia,37.292233,-121.881279,3
3,Townsville,Queensland,Australia,Oceania,34.125946,-118.291016,4
4,Townsville,Queensland,Australia,Oceania,18.253769,-66.037048,5
...,...,...,...,...,...,...,...
64862,Ho Chi Minh City,Ciudad Ho Chi Minh,Vietnam,Southeast Asia,18.209421,-66.370560,64863
64863,Ezhou,Hubei,China,Eastern Asia,39.935619,-83.138512,64864
64864,Tawau,Sabah,Malasia,Southeast Asia,33.710079,-84.112473,64865
64865,Semarang,Java Central,Indonesia,Southeast Asia,18.227577,-66.043983,64866


In [29]:
fact_order_items.head(10)

,order_id,customer_id,product_id,sales,quantity,discount,profit,item_total,profit_ratio,order_date,profit_outlier,profit_capped,profit_margin_pct,avg_selling_price,dicount_pct,order_month,quarter,weekday,is_weekend
0,77202,20755,1360,327.75,1,13.110000,91.250000,314.640015,0.29,2018-01-31 22:56:00,0,91.250000,27.841342,327.75,4.000000,1,1,Wednesday,0
1,75937,19490,1360,327.75,1,22.940001,22.860001,304.809998,0.08,2018-01-13 11:45:00,0,22.860001,6.974829,327.75,6.999237,1,1,Saturday,1
2,75936,19489,1360,327.75,1,29.500000,134.210007,298.250000,0.45,2018-01-13 11:24:00,0,134.210007,40.948896,327.75,9.000763,1,1,Saturday,1
3,75935,19488,1360,327.75,1,32.779999,18.580000,294.980011,0.06,2018-01-13 11:03:00,0,18.580000,5.668955,327.75,10.001525,1,1,Saturday,1
4,75934,19487,1360,327.75,1,39.330002,95.180000,288.420013,0.33,2018-01-13 10:42:00,0,95.180000,29.040427,327.75,12.000001,1,1,Saturday,1
5,75933,19486,1360,327.75,1,42.610001,68.430000,285.140015,0.24,2018-01-13 10:21:00,0,68.430000,20.878719,327.75,13.000763,1,1,Saturday,1
6,75932,19485,1360,327.75,1,49.160000,133.720001,278.589996,0.48,2018-01-13 10:00:00,0,133.720001,40.799390,327.75,14.999237,1,1,Saturday,1
7,75931,19484,1360,327.75,1,52.439999,132.149994,275.309998,0.48,2018-01-13 09:39:00,0,132.149994,40.320364,327.75,16.000000,1,1,Saturday,1
8,75930,19483,1360,327.75,1,55.720001,130.580002,272.029999,0.48,2018-01-13 09:18:00,0,130.580002,39.841343,327.75,17.000763,1,1,Saturday,1
9,75929,19482,1360,327.75,1,59.000000,45.689999,268.760010,0.17,2018-01-13 08:57:00,0,45.689999,13.940503,327.75,18.001526,1,1,Saturday,1


In [28]:
sales_by_region = pd.read_sql('''SELECT
    dl.'Order Region',
    SUM(f.sales) AS total_sales

FROM fact_order_items f

JOIN dim_location dl
ON f.location_id = dl.location_id

GROUP BY dl.'Order Region'
ORDER BY total_sales DESC;''',conn)

DatabaseError: Execution failed on sql 'SELECT
    dl.'Order Region',
    SUM(f.sales) AS total_sales

FROM fact_order_items f

JOIN dim_location dl
ON f.location_id = dl.location_id

GROUP BY dl.'Order Region'
ORDER BY total_sales DESC;': no such column: f.location_id

# shipping_performance

In [18]:
dim_shipping_wo_dup = pd.read_sql('''select * from dim_shipping_1_wo_dup''',conn)

In [15]:
dim_shipping.count()

shipping_id                      26
Shipping Mode                    26
Delivery Status                  26
Late_delivery_risk               26
Days for shipping (real)         26
Days for shipment (scheduled)    26
delievery_delay                  26
dtype: int64

In [20]:
shipping_performance_new = pd.read_sql('''SELECT
    "Shipping Mode",

    AVG("Days for shipping (real)" - "Days for shipment (scheduled)") AS avg_delay,

    SUM(CASE 
        WHEN ("Days for shipping (real)" - "Days for shipment (scheduled)") = 0 
        THEN 1 ELSE 0 
    END) * 1.0 / COUNT(*) AS on_time_rate,

    SUM(CASE 
        WHEN ("Days for shipping (real)" - "Days for shipment (scheduled)") > 0 
        THEN 1 ELSE 0 
    END) * 1.0 / COUNT(*) AS late_rate,

    SUM(CASE 
        WHEN ("Days for shipping (real)" - "Days for shipment (scheduled)") < 0 
        THEN 1 ELSE 0 
    END) * 1.0 / COUNT(*) AS early_rate,

    MIN("Days for shipping (real)" - "Days for shipment (scheduled)") AS best_case,

    MAX("Days for shipping (real)" - "Days for shipment (scheduled)") AS worst_case,

    COUNT(*) AS total_shipments

FROM dim_shipping_1_wo_dup
GROUP BY "Shipping Mode";''',conn)

In [24]:
shipping_performance_new .head()

,Shipping Mode,avg_delay,on_time_rate,late_rate,early_rate,best_case,worst_case,total_shipments
0,First Class,1.000000,0.000000,1.000000,0.000000,1,1,27814
1,Same Day,0.478279,0.521721,0.478279,0.000000,0,1,9737
2,Second Class,1.990828,0.202692,0.797308,0.000000,0,4,35216
3,Standard Class,-0.004093,0.199857,0.397682,0.402461,-2,2,107752


In [25]:
shipping_performance_new.to_sql(
    "shipping_performance_new",
    conn,
    if_exists="replace",
    index=False
)

4

# shipping_performace_new_wo_grouping_by

In [ ]:
Retail_main

In [7]:
order_delivery_performance = pd.read_sql('''
SELECT
    "Order Id" AS order_id,
    "Order Region" AS order_region,
    "Shipping Mode" AS shipping_mode,
    "Order Profit Per Order" AS profit,
    ("Days for shipping (real)" - "Days for shipment (scheduled)") AS delay_days,
    CASE 
        WHEN ("Days for shipping (real)" - "Days for shipment (scheduled)") > 0 THEN 1 
        ELSE 0 
    END AS is_late,
    CASE 
        WHEN ("Days for shipping (real)" - "Days for shipment (scheduled)") = 0 THEN 1 
        ELSE 0 
    END AS is_on_time,
    CASE 
        WHEN ("Days for shipping (real)" - "Days for shipment (scheduled)") < 0 THEN 1 
        ELSE 0 
    END AS is_early
FROM RetailIQ360
''', conn)

print(order_delivery_performance.shape)
print(order_delivery_performance.head())

(180519, 8)
   order_id    order_region   shipping_mode      profit  delay_days  is_late  \
0     77202  Southeast Asia  Standard Class   91.250000          -1        0   
1     75939      South Asia  Standard Class -249.089996           1        1   
2     75938      South Asia  Standard Class -247.779999           0        0   
3     75937         Oceania  Standard Class   22.860001          -1        0   
4     75936         Oceania  Standard Class  134.210007          -2        0   

   is_on_time  is_early  
0           0         1  
1           0         0  
2           1         0  
3           0         1  
4           0         1  


order_delivery_performance.to_sql(
    "shipping_performace_new_wo_grouping_by",
    conn,
    if_exists="replace",
    index=False
)

# creating fact_order_items_3_finals_for_ope

In [17]:
fact_order_items_3_finals = pd.read_sql('''
SELECT
    "Order Id" AS order_id,
    "Order Customer Id" AS customer_id,
    "Product Card Id" AS product_id,
    "Sales" AS sales,
    "Order Item Quantity" AS quantity,
    "Order Item Discount" AS discount,
    "Order Profit Per Order" AS profit,
    "Order Item Total" AS item_total,
    "Order Item Profit Ratio" AS profit_ratio,
    "order date (DateOrders)" AS order_date,
    "Order City" AS order_city,
    "Order State" AS order_state,
    "Order Country" AS order_country,
    "Order Region" AS order_region,
    "Order Zipcode" AS location_id,
    "Shipping Mode" AS shipping_mode,
    "Order Status" AS order_status,
    "Delivery Status" AS delivery_status,
    ("Days for shipping (real)" - "Days for shipment (scheduled)") AS delay_days,
    CASE WHEN ("Days for shipping (real)" - "Days for shipment (scheduled)") > 0 THEN 1 ELSE 0 END AS is_late,
    CASE WHEN ("Days for shipping (real)" - "Days for shipment (scheduled)") = 0 THEN 1 ELSE 0 END AS is_on_time,
    CASE WHEN ("Days for shipping (real)" - "Days for shipment (scheduled)") < 0 THEN 1 ELSE 0 END AS is_early
FROM RetailIQ360
''', conn)

print(fact_order_items_3_finals['delivery_status'].unique())
print(fact_order_items_3_finals.shape)

<ArrowStringArray>
['Advance shipping', 'Late delivery', 'Shipping on time', 'Shipping canceled']
Length: 4, dtype: str
(180519, 22)


In [18]:
# Make sure order_date is datetime
fact_order_items_3_finals['order_date'] = pd.to_datetime(fact_order_items_3_finals['order_date'])

# Keep rows where year is NOT 2018
fact_order_items_3_finals = fact_order_items_3_finals[
    fact_order_items_3_finals['order_date'].dt.year != 2018
]

In [16]:
fact_order_items_3_finals.to_sql(
    "creating fact_order_items_3_finals_for_ope",
    conn,
    if_exists="replace",
    index=False
)

178396

In [26]:
print(fact_order_items_3_finals[fact_order_items_3_finals['shipping_mode']=='First Class'][['delay_days']].describe())

       delay_days
count     27486.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0


# Creating Executive level kpis as view

In [43]:
conn.execute('''CREATE VIEW vw_executive_kpis AS

SELECT

    SUM(sales) AS total_revenue,

    SUM(profit) AS total_profit,

    AVG(profit_margin_pct) AS avg_margin,

    COUNT(DISTINCT customer_id)
        AS total_customers,

    COUNT(DISTINCT order_id)
        AS total_orders

FROM fact_order_items_enriched;''',conn)

OperationalError: view vw_executive_kpis already exists

In [44]:
conn.commit()

In [45]:
# SAVE ANALYTICAL TABLES

top_10_Cus_by_sales.to_sql(
    "top_10_Cus_by_sales",
    conn,
    if_exists="replace",
    index=False
)

profitability_analysis.to_sql(
    "profitability_analysis",
    conn,
    if_exists="replace",
    index=False
)
rank_products_desc.to_sql(
    "rank_products_desc",
    conn,
    if_exists="replace",
    index=False
)
cus_order_frequency.to_sql(
    "cus_order_frequency",
    conn,
    if_exists="replace",
    index=False
)

high_value_customers.to_sql(
    "high_value_customers",
    conn,
    if_exists="replace",
    index=False
)

com_daily_sales.to_sql(
    "com_daily_sales",
    conn,
    if_exists="replace",
    index=False
)

moving_avg_7d.to_sql(
    "moving_avg_7d",
    conn,
    if_exists="replace",
    index=False
)

pareto_analysis_table.to_sql(
    "pareto_analysis_table",
    conn,
    if_exists="replace",
    index=False
)
cohort_table.to_sql(
    "cohort_table",
    conn,
    if_exists="replace",
    index=False
)

# SAVE UPDATED DIM TABLES

shipping_performance.to_sql(
    "shipping_performance",
    conn,
    if_exists="replace",
    index=False
)





# FINALIZE DATABASE WRITE

conn.commit()

# CLOSE CONNECTION

conn.close()

print("All analytical tables saved successfully.")

All analytical tables saved successfully.
